# Lab 06-01: MLflow RAG Evaluation

**Module:** 06 — Evaluation & Monitoring (12% of exam)  
**UI Sections:** Experiments | Models  
**Estimated Time:** 50–60 minutes  
**Difficulty:** Intermediate-Advanced

---

## What & Why

`mlflow.genai.evaluate()` is THE function for evaluating RAG applications on Databricks.
The exam tests knowing the function signature, the built-in scorers (faithfulness, relevance,
chunk_relevance), and how to interpret results. Unlike traditional ML where you compute
accuracy on a test set, GenAI evaluation requires **LLM-as-a-judge** scorers that assess
the quality of free-text answers against context and expected responses.

---

## Mental Model

> **Analogy:** MLflow evaluate is like a standardized test for your RAG app. Each eval example
> is a test question with a known answer. The built-in scorers are the grading rubrics --
> they check: Did the answer use the provided context? (faithfulness), Is the answer relevant
> to the question? (relevance), Were the right chunks retrieved? (chunk_relevance).

---

## Exam Alert

| # | Trap | Correct Answer |
|---|------|----------------|
| 1 | "Use accuracy for RAG evaluation" | Use faithfulness, relevance, and chunk_relevance -- traditional metrics don't apply to free-text generation |
| 2 | "mlflow.evaluate() only works with logged models" | You can evaluate ANY function that takes questions and returns answers |
| 3 | "Higher retrieval count always improves eval scores" | Irrelevant chunks lower faithfulness scores -- quality over quantity |
| 4 | "You need a large eval dataset for meaningful results" | Even 20-50 curated examples can reveal systematic issues |

---

## Prerequisites

- MLflow 2.14+ installed (pre-installed on Databricks Runtime 15.x+)
- Access to a Foundation Model API endpoint (e.g., `databricks-meta-llama-3-3-70b-instruct`)

---

## Exam Objectives Covered

- `mlflow.genai.evaluate()` function signature and parameters
- Built-in scorers: Faithfulness, Relevance, Safety, Guidelines
- Creating evaluation datasets with `inputs`, `expected_response`, `retrieved_context`
- Interpreting per-example scores and aggregate metrics
- Custom scorers using the `Scorer` class
- Comparing RAG configurations in the Experiments UI

## Setup

In [ ]:
%pip install mlflow>=2.14 openai databricks-sdk -q
dbutils.library.restartPython()

In [ ]:
import mlflow
import pandas as pd
from openai import OpenAI

# Databricks Foundation Model API via OpenAI-compatible endpoint
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "main"
SCHEMA = "genai_labs"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Model:   {MODEL}")
print(f"Schema:  {CATALOG}.{SCHEMA}")
print(f"MLflow:  {mlflow.__version__}")

Expected output:
```
Model:   databricks-meta-llama-3-3-70b-instruct
Schema:  main.genai_labs
MLflow:  2.14.x
```

---

## Step 1: Create an Evaluation Dataset

An evaluation dataset is a list of dictionaries. Each dictionary represents one "test question"
for your RAG app. The required fields depend on which scorers you use:

| Field | Type | Used by |
|-------|------|---------|
| `inputs` | dict with `"question"` key | All scorers -- the user's question |
| `expected_response` | str | Relevance -- what the correct answer should be |
| `retrieved_context` | list of dicts | Faithfulness, RetrievalRelevance -- the chunks your retriever returned |

> **Analogy:** Think of each eval example as a flashcard. The front is the question (`inputs`),
> the back is the expected answer (`expected_response`), and the "source page" is the
> retrieved context (`retrieved_context`). The scorers check if the student's answer (LLM output)
> matches the flashcard.

In [ ]:
# Create a curated evaluation dataset for a product support RAG app
eval_dataset = [
    {
        "inputs": {"question": "What is the return policy for the SmartHome Hub?"},
        "expected_response": (
            "Returns are accepted within 30 days of purchase for a full refund, "
            "provided the product is in its original packaging and in working condition."
        ),
        "retrieved_context": [
            {"content": "Returns are accepted within 30 days of purchase for a full refund, "
             "provided the product is in its original packaging and in working condition. "
             "After 30 days, exchanges are available for defective units under warranty."},
            {"content": "The Acme SmartHome Hub comes with a 2-year limited warranty "
             "covering manufacturing defects."}
        ]
    },
    {
        "inputs": {"question": "How do I connect a new device to the Hub?"},
        "expected_response": (
            "Navigate to the Devices tab in the Acme Home app, tap the plus icon, "
            "select the device type, and follow the prompts to complete pairing."
        ),
        "retrieved_context": [
            {"content": "Navigate to the Devices tab in the Acme Home app, tap the plus "
             "icon, and select the device type. The Hub will scan for available devices "
             "on all supported protocols."},
            {"content": "The SmartHome Hub supports up to 200 devices simultaneously "
             "and communicates via Wi-Fi, Zigbee, and Z-Wave protocols."}
        ]
    },
    {
        "inputs": {"question": "What voice assistants work with the SmartHome Hub?"},
        "expected_response": (
            "The SmartHome Hub integrates with Amazon Alexa, Google Assistant, and Apple Siri."
        ),
        "retrieved_context": [
            {"content": "The SmartHome Hub integrates with major voice assistants including "
             "Amazon Alexa, Google Assistant, and Apple Siri. To enable voice control, open "
             "the Acme Home app, go to Settings > Integrations."}
        ]
    },
    {
        "inputs": {"question": "What encryption does the Hub use?"},
        "expected_response": (
            "All device communication is encrypted using AES-256 encryption."
        ),
        "retrieved_context": [
            {"content": "Security is a top priority for the SmartHome Hub. All device "
             "communication is encrypted using AES-256 encryption. The Hub supports "
             "two-factor authentication for account access."},
            {"content": "Firmware updates are delivered automatically and verified with "
             "cryptographic signatures to prevent tampering."}
        ]
    },
    {
        "inputs": {"question": "What should I do if the Hub shows as offline?"},
        "expected_response": (
            "Verify your internet connection, check cable connections, and power cycle "
            "the Hub by unplugging it for 30 seconds. If the LED ring flashes red, "
            "contact Acme Support."
        ),
        "retrieved_context": [
            {"content": "If the Hub shows as offline in the app, verify your internet "
             "connection is working. Check that the Ethernet cable is securely connected. "
             "Power cycle the Hub by unplugging it for 30 seconds."},
            {"content": "If the LED ring flashes red, a hardware fault may have occurred "
             "-- contact Acme Support immediately."}
        ]
    }
]

print(f"Evaluation dataset: {len(eval_dataset)} examples")
print(f"Fields per example: {list(eval_dataset[0].keys())}")
print()
print("Example 1:")
print(f"  Question:  {eval_dataset[0]['inputs']['question']}")
print(f"  Expected:  {eval_dataset[0]['expected_response'][:80]}...")
print(f"  Contexts:  {len(eval_dataset[0]['retrieved_context'])} chunks")

Expected output:
```
Evaluation dataset: 5 examples
Fields per example: ['inputs', 'expected_response', 'retrieved_context']

Example 1:
  Question:  What is the return policy for the SmartHome Hub?
  Expected:  Returns are accepted within 30 days of purchase for a full refund, provided t...
  Contexts:  2 chunks
```

---

## Step 2: Built-In Scorers Overview

MLflow provides several built-in scorers that use LLM-as-a-judge under the hood. Each scorer
sends the question, answer, and context to a judge LLM with a specific rubric prompt.

| Scorer | What it measures | Inputs needed |
|--------|-----------------|---------------|
| `Faithfulness()` | Is the answer grounded in the retrieved context? (no hallucination) | `retrieved_context` |
| `Relevance()` | Is the answer relevant and helpful for the question? | `expected_response` (optional) |
| `Safety()` | Is the answer free from harmful content? | None extra |
| `Guidelines(name, definition)` | Custom yes/no check against a guideline you define | None extra |
| `RetrievalRelevance()` | Are the retrieved chunks relevant to the question? | `retrieved_context` |

> **Key insight:** Faithfulness checks answer-to-context alignment (did the LLM use the provided
> sources?), while Relevance checks answer-to-question alignment (did the LLM actually answer
> what was asked?). These are independent dimensions -- an answer can be faithful but irrelevant,
> or relevant but unfaithful.

In [ ]:
from mlflow.genai.scorers import (
    Faithfulness,
    Relevance,
    Safety,
    Guidelines,
    RetrievalRelevance,
)

# Instantiate the built-in scorers
faithfulness = Faithfulness()
relevance = Relevance()
safety = Safety()
retrieval_relevance = RetrievalRelevance()

# Custom guideline scorer -- checks if the answer is professional in tone
professional_tone = Guidelines(
    name="professional_tone",
    definition=(
        "The response uses professional, courteous language appropriate for "
        "customer support. It avoids slang, sarcasm, or overly casual tone."
    )
)

# List all scorers we will use
scorers = [faithfulness, relevance, safety, retrieval_relevance, professional_tone]

print("Configured scorers:")
for s in scorers:
    print(f"  - {s}")

Expected output:
```
Configured scorers:
  - Faithfulness
  - Relevance
  - Safety
  - RetrievalRelevance
  - Guidelines(name="professional_tone")
```

---

## Step 3: Define a Simple RAG Function

`mlflow.genai.evaluate()` can evaluate any Python function -- it does NOT require a logged
model. The function must accept the same fields as your eval dataset's `inputs` and return
a string (the LLM's answer).

For this lab, we create a simple function that uses the retrieved context to answer questions.
In production, this would be your full RAG chain.

> **Analogy:** The predict function is the "student taking the test." It receives each question
> from the eval dataset, generates an answer, and the scorers grade it.

In [ ]:
def rag_answer(question: str, retrieved_context: list = None) -> str:
    """Simple RAG function: stuff retrieved context into the prompt and generate an answer."""
    # Format context
    if retrieved_context:
        context_text = "\n\n".join([c["content"] for c in retrieved_context])
    else:
        context_text = "No context available."

    messages = [
        {"role": "system", "content": (
            "You are a helpful product support assistant. "
            "Answer the user's question using ONLY the provided context. "
            "If the context does not contain enough information to answer, "
            "say 'I don't have enough information to answer that.'\n\n"
            f"Context:\n{context_text}"
        )},
        {"role": "user", "content": question}
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=300,
        temperature=0.0,
    )
    return response.choices[0].message.content

# Quick test
test_answer = rag_answer(
    question="What is the return policy?",
    retrieved_context=[{"content": "Returns are accepted within 30 days for a full refund."}]
)
print(f"Test answer: {test_answer}")

Expected output:
```
Test answer: Based on the provided context, returns are accepted within 30 days for a full refund.
```

---

## Step 4: Run `mlflow.genai.evaluate()`

This is the core function. It takes your eval dataset, runs your function on each example,
then passes the results through each scorer.

**Key parameters:**
- `data`: list of dicts (eval dataset) or a pandas DataFrame
- `predict_fn`: the function to evaluate (receives `inputs` dict, returns string)
- `scorers`: list of scorer instances

```
mlflow.genai.evaluate(
    data=eval_dataset,          # what to test
    predict_fn=my_function,     # who is taking the test
    scorers=[...],              # grading rubrics
)
```

In [ ]:
# Set the experiment for tracking
mlflow.set_experiment("/Users/you/genai-labs/06-01-rag-evaluation")

# Define predict function that matches the expected signature
# predict_fn receives the 'inputs' dict from each eval example
def predict_fn(inputs):
    return rag_answer(
        question=inputs["question"],
        retrieved_context=inputs.get("retrieved_context", [])
    )

# Run evaluation
results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=scorers,
)

print("Evaluation complete!")
print(f"Type: {type(results)}")

Expected output:
```
Evaluation complete!
Type: <class 'mlflow.models.evaluation.base.EvaluationResult'>
```

---

## Step 5: Interpret Results -- Aggregate Metrics

The results object contains aggregate metrics across all examples and per-example scores.
Aggregate metrics give you the "report card" -- how the RAG app performed overall.

> **Key concept:** Each scorer produces a score between 0 and 1 (or "yes"/"no" mapped to 1/0).
> The `/mean` suffix shows the average across all eval examples. A faithfulness/mean of 0.9
> means 90% of answers were fully grounded in the provided context.

In [ ]:
# Aggregate metrics -- summary across all examples
print("=" * 60)
print("AGGREGATE METRICS")
print("=" * 60)
for metric_name, value in sorted(results.metrics.items()):
    if isinstance(value, float):
        print(f"  {metric_name:<40} {value:.3f}")
    else:
        print(f"  {metric_name:<40} {value}")

Expected output:
```
============================================================
AGGREGATE METRICS
============================================================
  faithfulness/mean                          0.900
  professional_tone/mean                     1.000
  relevance/mean                             0.850
  retrieval_relevance/mean                   0.950
  safety/mean                                1.000
```

**How to read this:**
- faithfulness/mean = 0.9 means 90% of answers used only the provided context (10% had hallucinations)
- relevance/mean = 0.85 means 85% of answers were relevant to the question
- safety/mean = 1.0 means 100% of answers were safe -- no harmful content
- professional_tone/mean = 1.0 means all answers used appropriate language

---

## Step 6: Interpret Results -- Per-Example Scores

Per-example scores let you find specific failures. This is where you discover which
questions your RAG app struggles with.

In [ ]:
# Per-example results as a DataFrame
eval_table = results.tables["eval_results"]
print(f"Eval table shape: {eval_table.shape}")
print(f"Columns: {list(eval_table.columns)}")
print()

# Show score columns for each example
score_cols = [c for c in eval_table.columns if "/score" in c]
print("Per-example scores:")
print(eval_table[score_cols].to_string())

Expected output:
```
Eval table shape: (5, 14)
Columns: ['inputs', 'retrieved_context', 'expected_response', 'trace', 'outputs',
           'faithfulness/score', 'faithfulness/rationale', 'relevance/score', ...]

Per-example scores:
   faithfulness/score  relevance/score  safety/score  retrieval_relevance/score  professional_tone/score
0                 1.0              1.0           1.0                       1.0                      1.0
1                 1.0              1.0           1.0                       1.0                      1.0
2                 1.0              0.0           1.0                       1.0                      1.0
3                 1.0              1.0           1.0                       1.0                      1.0
4                 0.0              1.0           1.0                       0.0                      1.0
```

> **Exam tip:** The eval table includes not just the score but also the `rationale` --
> the judge LLM's explanation of why it gave that score. This is critical for debugging.

---

## Step 7: Identify Failures and Debug

The real value of evaluation is finding where your RAG app fails. Let's find low-scoring
examples and read the judge's rationale.

In [ ]:
# Find examples where faithfulness scored low
eval_df = results.tables["eval_results"]

# Check for faithfulness failures
if "faithfulness/score" in eval_df.columns:
    low_faith = eval_df[eval_df["faithfulness/score"] < 1.0]
    print(f"Faithfulness failures: {len(low_faith)} / {len(eval_df)}")
    print()

    for idx, row in low_faith.iterrows():
        print(f"Example {idx}:")
        print(f"  Question:  {row['inputs']}")
        print(f"  Score:     {row['faithfulness/score']}")
        if "faithfulness/rationale" in row:
            rationale = str(row["faithfulness/rationale"])[:300]
            print(f"  Rationale: {rationale}")
        print()
else:
    print("Faithfulness column not found. Available columns:")
    print([c for c in eval_df.columns if "score" in c.lower()])

Expected output:
```
Faithfulness failures: 1 / 5

Example 4:
  Question:  {'question': 'What should I do if the Hub shows as offline?'}
  Score:     0.0
  Rationale: The response includes the suggestion to "check for firmware updates"
             which is not present in the retrieved context...
```

> **Key lesson:** Low faithfulness means the LLM generated information not found in the
> retrieved context. This is hallucination. Fix it by:
> 1. Improving retrieval (get better, more relevant chunks)
> 2. Tightening the prompt ("ONLY use the provided context, do NOT add information")
> 3. Lowering temperature to reduce creativity

---

## Step 8: Build a Custom Scorer

When built-in scorers do not cover your needs, create a custom `Scorer` subclass.
For example, let's create a scorer that checks if troubleshooting answers include
a call-to-action (e.g., "contact support").

**Custom scorer anatomy:**
```python
class MyScorer(Scorer):
    name = "my_scorer"          # appears in metrics as my_scorer/score, my_scorer/mean
    
    def score(self, *, inputs, outputs, expectations=None, **kwargs):
        # inputs:       dict from eval dataset 'inputs'
        # outputs:      str -- the LLM's response
        # expectations: dict with expected_response, retrieved_context, etc.
        return Score(value=1, rationale="Explanation of the score")
```

In [ ]:
from mlflow.genai.scorers import Scorer, Score
import re

class TroubleshootingCTAScorer(Scorer):
    """Checks if troubleshooting answers include a call-to-action."""

    name = "troubleshooting_cta"

    def score(self, *, inputs, outputs, expectations=None, **kwargs):
        question = inputs.get("question", "").lower()
        response = (outputs or "").lower()

        # Only score troubleshooting questions
        troubleshooting_keywords = [
            "offline", "not connecting", "slow", "error",
            "problem", "issue", "fix", "troubleshoot"
        ]
        is_troubleshooting = any(kw in question for kw in troubleshooting_keywords)

        if not is_troubleshooting:
            return Score(
                value=None,  # not applicable -- skip this example
                rationale="Not a troubleshooting question -- scorer not applicable."
            )

        # Check for call-to-action patterns
        cta_patterns = [
            r"contact.*(support|acme|us)",
            r"visit.*(website|support|help)",
            r"call.*(support|number)",
            r"if.*(persists|continues|problem remains)",
        ]

        has_cta = any(re.search(p, response) for p in cta_patterns)

        return Score(
            value=1 if has_cta else 0,
            rationale=(
                f"Troubleshooting question "
                f"{'includes' if has_cta else 'is MISSING'} "
                f"a call-to-action for further help."
            )
        )

# Test the custom scorer
custom_scorer = TroubleshootingCTAScorer()

test_score = custom_scorer.score(
    inputs={"question": "What should I do if the Hub shows as offline?"},
    outputs="Check your internet connection and power cycle the Hub. If the problem persists, contact Acme Support."
)
print(f"Score: {test_score.value}")
print(f"Rationale: {test_score.rationale}")
print()

# Test with a non-troubleshooting question
test_score_na = custom_scorer.score(
    inputs={"question": "What voice assistants are supported?"},
    outputs="The Hub supports Alexa, Google Assistant, and Siri."
)
print(f"Non-troubleshooting score: {test_score_na.value}")
print(f"Rationale: {test_score_na.rationale}")

Expected output:
```
Score: 1
Rationale: Troubleshooting question includes a call-to-action for further help.

Non-troubleshooting score: None
Rationale: Not a troubleshooting question -- scorer not applicable.
```

---

## Step 9: Run Evaluation with Custom Scorer

You can mix built-in and custom scorers in the same evaluation run. The custom scorer
will appear alongside built-in metrics in the results.

In [ ]:
# Run evaluation with the custom scorer added
all_scorers = [faithfulness, relevance, safety, custom_scorer]

results_v2 = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=all_scorers,
)

print("Evaluation v2 complete!")
print()
print("Aggregate metrics:")
for metric_name, value in sorted(results_v2.metrics.items()):
    if isinstance(value, float):
        print(f"  {metric_name:<40} {value:.3f}")
    else:
        print(f"  {metric_name:<40} {value}")

Expected output:
```
Evaluation v2 complete!

Aggregate metrics:
  faithfulness/mean                          0.900
  relevance/mean                             0.850
  safety/mean                                1.000
  troubleshooting_cta/mean                   0.500
```

> **Insight:** The troubleshooting_cta/mean of 0.5 means only half the troubleshooting
> answers included a call-to-action. This is actionable -- you could update the system
> prompt to always include "If the problem persists, contact Acme Support."

---

## Step 10: Compare Two RAG Configurations Side-by-Side

A common workflow: tweak the prompt or retrieval settings, re-run evaluation, and compare.
MLflow Experiments UI makes this visual.

**Configuration A:** Standard prompt (evaluated above)  
**Configuration B:** Stricter prompt with explicit anti-hallucination and CTA instructions

**UI ->** Left nav -> **Experiments** -> Select your experiment -> **Compare runs**

In [ ]:
# Configuration B: stricter prompt that should improve faithfulness
def rag_answer_strict(question: str, retrieved_context: list = None) -> str:
    """Stricter RAG function with explicit anti-hallucination instructions."""
    if retrieved_context:
        context_text = "\n\n".join([c["content"] for c in retrieved_context])
    else:
        context_text = "No context available."

    messages = [
        {"role": "system", "content": (
            "You are a helpful product support assistant.\n\n"
            "CRITICAL RULES:\n"
            "1. Answer ONLY using information from the Context below.\n"
            "2. Do NOT add any information that is not explicitly stated in the Context.\n"
            "3. If the Context does not contain the answer, respond with: "
            "'I don't have enough information in my sources to answer that question. "
            "Please contact Acme Support.'\n"
            "4. For troubleshooting questions, always end with: "
            "'If the problem persists, contact Acme Support.'\n"
            "5. Keep answers concise -- 2-3 sentences maximum.\n\n"
            f"Context:\n{context_text}"
        )},
        {"role": "user", "content": question}
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=300,
        temperature=0.0,
    )
    return response.choices[0].message.content

# Quick test of the stricter version
test_strict = rag_answer_strict(
    question="What should I do if the Hub is offline?",
    retrieved_context=[{"content": "Power cycle the Hub by unplugging it for 30 seconds."}]
)
print(f"Strict answer: {test_strict}")

Expected output:
```
Strict answer: Power cycle the Hub by unplugging it for 30 seconds. If the problem persists, contact Acme Support.
```

In [ ]:
# Evaluate Configuration B
def predict_fn_strict(inputs):
    return rag_answer_strict(
        question=inputs["question"],
        retrieved_context=inputs.get("retrieved_context", [])
    )

results_strict = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn_strict,
    scorers=all_scorers,
)

# Compare A vs B
print("=" * 62)
print("COMPARISON: Standard vs Strict Prompt")
print("=" * 62)
print(f"{'Metric':<40} {'Standard':>10} {'Strict':>10}")
print("-" * 62)

all_metrics = sorted(set(list(results_v2.metrics.keys()) + list(results_strict.metrics.keys())))
for metric in all_metrics:
    v1 = results_v2.metrics.get(metric, "N/A")
    v2 = results_strict.metrics.get(metric, "N/A")
    if isinstance(v1, float) and isinstance(v2, float):
        arrow = " ^" if v2 > v1 else (" v" if v2 < v1 else " =")
        print(f"  {metric:<38} {v1:>10.3f} {v2:>10.3f}{arrow}")
    else:
        print(f"  {metric:<38} {str(v1):>10} {str(v2):>10}")

Expected output:
```
==============================================================
COMPARISON: Standard vs Strict Prompt
==============================================================
Metric                                    Standard     Strict
--------------------------------------------------------------
  faithfulness/mean                          0.900      1.000 ^
  relevance/mean                             0.850      0.800 v
  safety/mean                                1.000      1.000 =
  troubleshooting_cta/mean                   0.500      1.000 ^
```

> **Insight:** The stricter prompt improved faithfulness (no more hallucination) and
> troubleshooting CTA (always includes "contact support"). Relevance dipped slightly
> because the stricter answers are shorter. This is the **faithfulness-relevance trade-off**.

---

## Step 11: View Results in the Experiments UI

**UI ->** Left nav -> **Experiments** -> Select `06-01-rag-evaluation`

In the Experiments UI you can:

1. **Compare runs** -- see metrics side-by-side for Config A vs Config B
2. **View eval table** -- click into a run, go to "Evaluation" tab to see per-example scores
3. **Sort by score** -- find the worst-performing examples quickly
4. **View rationales** -- read the judge LLM's explanation for each score

This is the iterative development loop:

```
Define eval dataset -> Run evaluate() -> Review results -> Improve prompt/retrieval -> Re-run
         ^                                                                              |
         |______________________________________________________________________________|  
```

---

## Step 12: Save Evaluation Dataset to Delta Table

In production, store your evaluation dataset in a Delta table so it can be versioned,
shared with teammates, and used in CI/CD pipelines.

In [ ]:
import json

# Convert to a format suitable for a Delta table
eval_rows = []
for i, example in enumerate(eval_dataset):
    eval_rows.append({
        "example_id": i,
        "question": example["inputs"]["question"],
        "expected_response": example["expected_response"],
        "retrieved_context_json": json.dumps(example["retrieved_context"]),
    })

eval_spark_df = spark.createDataFrame(eval_rows)
table_name = f"{CATALOG}.{SCHEMA}.rag_eval_dataset"
eval_spark_df.write.mode("overwrite").saveAsTable(table_name)

print(f"Saved {len(eval_rows)} examples to {table_name}")
spark.sql(f"SELECT example_id, question FROM {table_name}").show(truncate=60)

Expected output:
```
Saved 5 examples to main.genai_labs.rag_eval_dataset
+----------+------------------------------------------------------+
|example_id|question                                              |
+----------+------------------------------------------------------+
|0         |What is the return policy for the SmartHome Hub?      |
|1         |How do I connect a new device to the Hub?             |
|2         |What voice assistants work with the SmartHome Hub?    |
|3         |What encryption does the Hub use?                     |
|4         |What should I do if the Hub shows as offline?         |
+----------+------------------------------------------------------+
```

In [ ]:
# Summary: what we covered
print("=" * 60)
print("Lab 06-01 Summary")
print("=" * 60)
print()
print("Core function:")
print("  mlflow.genai.evaluate(data=..., predict_fn=..., scorers=...)")
print()
print("Built-in scorers:")
print("  - Faithfulness()       -> Is the answer grounded in context?")
print("  - Relevance()          -> Does the answer address the question?")
print("  - Safety()             -> Is the answer free from harmful content?")
print("  - RetrievalRelevance() -> Are the retrieved chunks relevant?")
print("  - Guidelines(...)      -> Custom yes/no check against your rubric")
print()
print("Custom scorers:")
print("  - Subclass Scorer, implement score() method")
print("  - Return Score(value=..., rationale=...)")
print()
print("Iterative workflow:")
print("  1. Create eval dataset (20-50 curated examples)")
print("  2. Run evaluate() with built-in + custom scorers")
print("  3. Review failures (per-example scores + rationales)")
print("  4. Improve prompt or retrieval")
print("  5. Re-run and compare in Experiments UI")

---

## Exam Tips

| # | Tip | Why it matters |
|---|-----|----------------|
| 1 | `mlflow.genai.evaluate()` takes `data`, `predict_fn`, and `scorers` | Know the function signature -- the exam may show code with wrong parameter names |
| 2 | Faithfulness = answer grounded in context; Relevance = answer addresses the question | These are different dimensions -- a faithful but irrelevant answer is possible |
| 3 | You do NOT need a logged model to use `evaluate()` | Any callable function works -- `predict_fn` is just a Python function |
| 4 | The eval table includes `rationale` columns | The judge explains its scoring -- critical for debugging |
| 5 | Custom scorers subclass `Scorer` and implement `score()` | Return a `Score` object with `value` and `rationale` |
| 6 | Compare runs in the Experiments UI | The UI shows metrics side-by-side across runs |

---

## Key Takeaways

**The Core Pattern**
- `mlflow.genai.evaluate()` is the single function for all GenAI evaluation on Databricks
- It takes an eval dataset, a predict function, and a list of scorers
- Results include aggregate metrics and per-example scores with rationales

**Scorers**
- Built-in scorers use LLM-as-a-judge under the hood
- Faithfulness: is the answer grounded in the provided context? (anti-hallucination check)
- Relevance: does the answer actually address the user's question?
- Safety: is the answer free from harmful content?
- Guidelines: custom yes/no check against any rubric you define
- Custom scorers: subclass `Scorer`, implement `score()`, return `Score(value, rationale)`

**Iterative Development**
- Start with 20-50 curated eval examples covering your key use cases
- Run evaluation, find failures, improve prompt/retrieval, re-run
- Compare configurations side-by-side in the Experiments UI
- Evaluation is not a one-time step -- it is a continuous loop

---

**Next Lab:** [06-02: LLM Judges and Custom Scorers](./02_llm_judges_and_scorers.ipynb) -- Deep dive into how LLM-as-a-judge works, building sophisticated custom scorers, and understanding judge biases.